In [1]:
import pandas as pd
import random
import math

In [38]:
# =========================================
# LOAD DATA
# =========================================

df = pd.read_csv("H4_cosine_similarity.csv")

In [39]:
# =========================================
# PASTIKAN FORMAT
# =========================================

df["cosine_similarity"] = (
    df["cosine_similarity"]
    .astype(float)
)

df["pertanyaan"] = (
    df["pertanyaan"]
    .astype(str)
)

In [40]:
# =========================================
# HAPUS DUPLIKAT GLOBAL
# AGAR PERTANYAAN TIDAK TERAMBIL 2X
# =========================================

df = df.drop_duplicates(
    subset=["pertanyaan"]
).reset_index(drop=True)

# =========================================
# TIPE PRIORITAS
# =========================================

priority_types = [
    "siapa",
    "apa",
    "di mana",
    "ke mana",
    "dari mana",
    "kapan"
]

In [41]:
# =========================================
# DETEKSI TIPE PERTANYAAN
# =========================================

def detect_question_type(q):

    q_lower = q.lower().strip()

    if q_lower.startswith("siapa"):
        return "siapa"

    elif q_lower.startswith("apa"):
        return "apa"

    elif q_lower.startswith("di mana"):
        return "di mana"

    elif q_lower.startswith("ke mana"):
        return "ke mana"

    elif q_lower.startswith("dari mana"):
        return "dari mana"

    elif q_lower.startswith("kapan"):
        return "kapan"

    return "lainnya"

In [42]:
# =========================================
# FLOOR 2 DESIMAL
# TANPA PEMBULATAN
# =========================================

def floor_2_decimal(x):

    return math.floor(x * 100) / 100

# =========================================
# TAMBAH TIPE
# =========================================

df["tipe"] = df["pertanyaan"].apply(
    detect_question_type
)

# =========================================
# TAMBAH SCORE FLOOR
# =========================================

df["score_floor"] = df[
    "cosine_similarity"
].apply(floor_2_decimal)

# =========================================
# RENTANG
# =========================================

ranges = [

    (0.90, 1.00),
    (0.80, 0.89),
    (0.70, 0.79),
    (0.60, 0.69),
    (0.50, 0.59),
    (0.40, 0.49),
    (0.30, 0.39),
    (0.20, 0.29),
    (0.10, 0.19),
    (0.01, 0.09)

]

# =========================================
# HASIL AKHIR
# =========================================

hasil = []

In [43]:
# =========================================
# LOOP SETIAP RENTANG
# =========================================

for start, end in ranges:

    print(f"\nPROSES RENTANG {start:.2f} - {end:.2f}")

    # =====================================
    # FILTER RENTANG
    # =====================================

    subset = df[
        (df["score_floor"] >= start)
        &
        (df["score_floor"] <= end)
    ].copy()

    # =====================================
    # UNIQUE SCORE DESC
    # =====================================

    unique_scores = sorted(
        subset["score_floor"].unique(),
        reverse=True
    )

    selected_rows = []

    used_indexes = set()

    used_questions = set()

    type_index = 0

    # =====================================
    # AMBIL SATU-SATU
    # DARI SCORE TERTINGGI
    # =====================================

    for score in unique_scores:

        if len(selected_rows) >= 10:
            break

        score_df = subset[
            subset["score_floor"] == score
        ]

        found = None

        # =================================
        # CARI BERDASARKAN URUTAN TIPE
        # =================================

        for offset in range(len(priority_types)):

            try_type = priority_types[
                (type_index + offset)
                % len(priority_types)
            ]

            candidates = score_df[
                (score_df["tipe"] == try_type)
                &
                (~score_df.index.isin(used_indexes))
                &
                (~score_df["pertanyaan"].isin(used_questions))
            ]

            if len(candidates) > 0:

                found = candidates.sample(
                    1,
                    random_state=random.randint(1,99999)
                )

                type_index = (
                    priority_types.index(try_type) + 1
                )

                break

        # =================================
        # JIKA TIDAK ADA
        # AMBIL RANDOM
        # =================================

        if found is None:

            remaining = score_df[
                (~score_df.index.isin(used_indexes))
                &
                (~score_df["pertanyaan"].isin(used_questions))
            ]

            if len(remaining) > 0:

                found = remaining.sample(
                    1,
                    random_state=random.randint(1,99999)
                )

        # =================================
        # SIMPAN
        # =================================

        if found is not None:

            idx = found.index[0]

            q = found.iloc[0]["pertanyaan"]

            used_indexes.add(idx)

            used_questions.add(q)

            selected_rows.append(found)

    # =====================================
    # JIKA BELUM 10
    # TAMBAH RANDOM
    # TANPA DUPLIKAT
    # =====================================

    while len(selected_rows) < 10:

        remaining = subset[
            (~subset.index.isin(used_indexes))
            &
            (~subset["pertanyaan"].isin(used_questions))
        ]

        if len(remaining) == 0:
            break

        extra = remaining.sample(
            1,
            random_state=random.randint(1,99999)
        )

        idx = extra.index[0]

        q = extra.iloc[0]["pertanyaan"]

        used_indexes.add(idx)

        used_questions.add(q)

        selected_rows.append(extra)

    # =====================================
    # GABUNGKAN
    # =====================================

    if len(selected_rows) > 0:

        final_subset = pd.concat(
            selected_rows
        )

        final_subset["rentang"] = (
            f"{start:.2f}-{end:.2f}"
        )

        hasil.append(final_subset)


PROSES RENTANG 0.90 - 1.00

PROSES RENTANG 0.80 - 0.89

PROSES RENTANG 0.70 - 0.79

PROSES RENTANG 0.60 - 0.69

PROSES RENTANG 0.50 - 0.59

PROSES RENTANG 0.40 - 0.49

PROSES RENTANG 0.30 - 0.39

PROSES RENTANG 0.20 - 0.29

PROSES RENTANG 0.10 - 0.19

PROSES RENTANG 0.01 - 0.09


In [44]:
# =========================================
# GABUNG SEMUA
# =========================================

df_final = pd.concat(
    hasil,
    ignore_index=True
)

# =========================================
# SORT
# =========================================

df_final = df_final.sort_values(
    by=[
        "rentang",
        "score_floor"
    ],
    ascending=[False, False]
)

# =========================================
# RESET INDEX
# =========================================

df_final = df_final.reset_index(drop=True)

In [46]:
# =========================================
# OUTPUT
# =========================================

print("\n")
print("=" * 60)
print("JUMLAH DATA AKHIR")
print("=" * 60)

print(len(df_final))

print("\nCONTOH:")

df_final[
    [
        "id_teks",
        "id_kalimat",
        "kalimat",
        "pertanyaan",
        "cosine_similarity"
    ]
].head(20)



JUMLAH DATA AKHIR
100

CONTOH:


,id_teks,id_kalimat,kalimat,pertanyaan,cosine_similarity
0,43,8,Siska mencari nomor kursi tiket.,Siapa yang mencari nomor kursi tiket?,0.9677
1,42,23,Doni membayangkan rasa tomat yang segar.,Siapa yang membayangkan rasa tomat yang segar?,0.9562
2,47,25,Adik mendengarkan cerita Nina dengan antusias.,Siapa yang mendengarkan cerita Nina dengan ant...,0.9542
3,91,22,Bentuknya menyerupai naga asli yang sedang ter...,Apa yang menyerupai naga asli yang sedang terb...,0.9417
4,87,7,Yoga mengambil dua baterai baru di laci.,Di mana Yoga mengambil dua baterai baru?,0.9378
5,97,21,Raka berterima kasih untuk kado alien tersebut.,Siapa yang berterima kasih untuk kado alien te...,0.9306
6,15,8,Budi menaiki sepeda dengan hati-hati.,Siapa yang menaiki sepeda dengan hati-hati?,0.9207
7,76,20,Tanduk itu berguna menyingkirkan ranting pohon...,Apa yang berguna menyingkirkan ranting pohon r...,0.9152
8,100,11,Bobi menambahkan kotak kuning di atas.,Di mana Bobi menambahkan kotak kuning?,0.9091
9,49,13,Lisa melihat kupu-kupu dari luar jaring.,Dari mana Lisa melihat kupu-kupu?,0.9010


In [ ]:
# =========================================
# SIMPAN
# =========================================

df_final.to_csv(
    "H4_CS_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFile berhasil disimpan!")